In [3]:
import pandas as pd
import numpy as np
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import pickle
from pathlib import Path
import os

# Check where we currently are
print("Current directory:", os.getcwd())

# Search for the file anywhere in the project
for root, dirs, files in os.walk("C:/Users/prakh/connect-ml"):
    for file in files:
        if file.endswith(".csv"):
            print("Found:", os.path.join(root, file))
print("✅ Imports working")

Current directory: C:\Users\prakh\connect-ml
Found: C:/Users/prakh/connect-ml\data\processed\jobs_with_domains.csv
Found: C:/Users/prakh/connect-ml\data\raw\job_skills.csv
Found: C:/Users/prakh/connect-ml\data\raw\postings.csv
Found: C:/Users/prakh/connect-ml\venv\Lib\site-packages\matplotlib\mpl-data\sample_data\data_x_x2_x3.csv
Found: C:/Users/prakh/connect-ml\venv\Lib\site-packages\matplotlib\mpl-data\sample_data\msft.csv
Found: C:/Users/prakh/connect-ml\venv\Lib\site-packages\matplotlib\mpl-data\sample_data\Stocks.csv
Found: C:/Users/prakh/connect-ml\venv\Lib\site-packages\numpy\random\tests\data\mt19937-testset-1.csv
Found: C:/Users/prakh/connect-ml\venv\Lib\site-packages\numpy\random\tests\data\mt19937-testset-2.csv
Found: C:/Users/prakh/connect-ml\venv\Lib\site-packages\numpy\random\tests\data\pcg64-testset-1.csv
Found: C:/Users/prakh/connect-ml\venv\Lib\site-packages\numpy\random\tests\data\pcg64-testset-2.csv
Found: C:/Users/prakh/connect-ml\venv\Lib\site-packages\numpy\random

In [4]:
df = pd.read_csv("C:/Users/prakh/connect-ml/data/processed/jobs_with_domains.csv")

print(f"Loaded {df.shape[0]} rows")
print(df['domain'].value_counts())

Loaded 5125 rows
domain
Software Engineering    1742
Data & AI               1178
Finance                  981
Marketing                540
Product Management       531
Design                   153
Name: count, dtype: int64


In [5]:
print("Loading model... this may take 1-2 minutes on first run")

model = SentenceTransformer('all-MiniLM-L6-v2')

print("✅ Model loaded")

Loading model... this may take 1-2 minutes on first run


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ Model loaded


In [6]:
from tqdm import tqdm

# Use first 500 chars of description
df['description_short'] = df['description'].str[:500].fillna('')

print(f"Creating embeddings for {len(df)} job descriptions...")
print("This will take 3-5 minutes, please wait...\n")

embeddings = model.encode(
    df['description_short'].tolist(),
    batch_size=64,
    show_progress_bar=True
)

print(f"\n✅ Done! Embedding shape: {embeddings.shape}")

Creating embeddings for 5125 job descriptions...
This will take 3-5 minutes, please wait...



Batches:   0%|          | 0/81 [00:00<?, ?it/s]


✅ Done! Embedding shape: (5125, 384)


In [8]:
import numpy as np

np.save("C:/Users/prakh/connect-ml/models/job_embeddings.npy", embeddings)
df.to_csv("C:/Users/prakh/connect-ml/data/processed/jobs_with_domains.csv", index=False)

print("✅ Embeddings saved")

✅ Embeddings saved


In [9]:
def predict_career_paths(
    student_skills: str,
    student_interests: str,
    target_domain: str,
    top_n: int = 3
):
    # Combine student profile into one text
    student_profile = f"Skills: {student_skills}. Interests: {student_interests}"
    
    # Convert to embedding
    student_embedding = model.encode([student_profile])
    
    # Filter to chosen domain only
    domain_mask = df['domain'] == target_domain
    domain_df = df[domain_mask].reset_index(drop=True)
    domain_embeddings = embeddings[domain_mask]
    
    if len(domain_df) == 0:
        return {"error": f"No jobs found for domain: {target_domain}"}
    
    # Calculate similarity scores
    similarities = cosine_similarity(student_embedding, domain_embeddings)[0]
    
    # Get top matches
    top_indices = similarities.argsort()[::-1][:top_n * 5]
    
    # Deduplicate by title
    seen_titles = set()
    results = []
    
    for idx in top_indices:
        title = domain_df.iloc[idx]['title'].title()
        score = round(float(similarities[idx]) * 100, 1)
        
        if title not in seen_titles:
            seen_titles.add(title)
            
            exp_level = domain_df.iloc[idx].get('formatted_experience_level', 'Not specified')
            if pd.isna(exp_level):
                exp_level = 'All levels'
                
            results.append({
                "career_path": title,
                "match_score": f"{score}%",
                "experience_level": exp_level,
                "domain": target_domain
            })
        
        if len(results) == top_n:
            break
    
    return results

print("✅ Predictor function ready")

✅ Predictor function ready


In [10]:
results = predict_career_paths(
    student_skills="Python, SQL, statistics, data visualization",
    student_interests="I love analyzing data, finding patterns, and building predictive models",
    target_domain="Data & AI",
    top_n=3
)

print("🎯 Career Path Predictions:\n")
for i, path in enumerate(results, 1):
    print(f"Option {i}:")
    print(f"  Role        : {path['career_path']}")
    print(f"  Match Score : {path['match_score']}")
    print(f"  Level       : {path['experience_level']}")
    print()

🎯 Career Path Predictions:

Option 1:
  Role        : Data Scientist Ii
  Match Score : 68.8%
  Level       : Entry level

Option 2:
  Role        : Data Analyst
  Match Score : 64.1%
  Level       : Mid-Senior level

Option 3:
  Role        : Data Analyst 3
  Match Score : 63.8%
  Level       : Mid-Senior level



In [11]:
import re

def clean_title(title):
    """Remove things like 'Ii', 'Iii', '2', '3' from the end of job titles."""
    # Remove trailing numbers
    title = re.sub(r'\s+\d+$', '', title.strip())
    # Remove trailing roman numerals (Ii, Iii, Iv)
    title = re.sub(r'\s+(Ii|Iii|Iv|Vi|Vii)$', '', title.strip())
    return title.strip()

# Test it
print(clean_title("Data Scientist Ii"))    # → Data Scientist
print(clean_title("Data Analyst 3"))       # → Data Analyst
print(clean_title("Software Engineer Iii"))# → Software Engineer

Data Scientist
Data Analyst
Software Engineer


In [12]:
def predict_career_paths(
    student_skills: str,
    student_interests: str,
    target_domain: str,
    top_n: int = 3
):
    student_profile = f"Skills: {student_skills}. Interests: {student_interests}"
    student_embedding = model.encode([student_profile])
    
    domain_mask = df['domain'] == target_domain
    domain_df = df[domain_mask].reset_index(drop=True)
    domain_embeddings = embeddings[domain_mask]
    
    if len(domain_df) == 0:
        return {"error": f"No jobs found for domain: {target_domain}"}
    
    similarities = cosine_similarity(student_embedding, domain_embeddings)[0]
    top_indices = similarities.argsort()[::-1][:top_n * 10]
    
    seen_titles = set()
    results = []
    
    for idx in top_indices:
        raw_title = domain_df.iloc[idx]['title'].title()
        title = clean_title(raw_title)
        score = round(float(similarities[idx]) * 100, 1)
        
        if title not in seen_titles:
            seen_titles.add(title)
            
            exp_level = domain_df.iloc[idx].get('formatted_experience_level', 'Not specified')
            if pd.isna(exp_level):
                exp_level = 'All levels'
                
            results.append({
                "career_path": title,
                "match_score": f"{score}%",
                "experience_level": exp_level,
                "domain": target_domain
            })
        
        if len(results) == top_n:
            break
    
    return results

print("✅ Updated predictor ready")

✅ Updated predictor ready


In [13]:
test_cases = [
    {
        "skills": "Python, SQL, statistics, data visualization",
        "interests": "analyzing data and building predictive models",
        "domain": "Data & AI"
    },
    {
        "skills": "JavaScript, React, HTML, CSS",
        "interests": "building websites and user interfaces",
        "domain": "Software Engineering"
    },
    {
        "skills": "communication, market research, Excel",
        "interests": "understanding customers and growing brands",
        "domain": "Marketing"
    }
]

for test in test_cases:
    print(f"\n{'='*45}")
    print(f"Domain: {test['domain']}")
    print(f"Skills: {test['skills']}")
    print(f"{'='*45}")
    
    results = predict_career_paths(
        student_skills=test['skills'],
        student_interests=test['interests'],
        target_domain=test['domain'],
        top_n=3
    )
    
    for i, path in enumerate(results, 1):
        print(f"  {i}. {path['career_path']} ({path['match_score']}) — {path['experience_level']}")


Domain: Data & AI
Skills: Python, SQL, statistics, data visualization
  1. Data Analyst (68.0%) — Mid-Senior level
  2. Data Scientist (66.6%) — Entry level
  3. Senior Data Analyst (62.9%) — All levels

Domain: Software Engineering
Skills: JavaScript, React, HTML, CSS
  1. Software Engineer (68.9%) — Mid-Senior level
  2. Frontend Engineer (66.4%) — All levels
  3. Lead / Sr. Software Engineer, React + Front End (61.9%) — Mid-Senior level

Domain: Marketing
Skills: communication, market research, Excel
  1. Marketing Manager (62.4%) — All levels
  2. Digital Marketing Specialist (59.7%) — Mid-Senior level
  3. Product Marketing Manager (58.6%) — All levels
